# Controlling MODFLOW 6 with the API — E. Streamflow augmentation at the coast

This is a variant of part E of the MODFLOW 6 application programming interface
(**API**) series — see [**A. Basic usage**](mf6-api-a-basic.ipynb) for the API
lifecycle and
[**C. Adjusting recharge with a callback**](mf6-api-c-recharge-callback.ipynb)
for the callback mechanism this example builds on.

[**E. Streamflow augmentation**](mf6-api-e-streamflow-augmentation.ipynb) runs an
operating rule on the inland valley: pump a well whenever flow at the stream
gauge falls below a threshold, and add the pumped water to the stream. The rule
runs, but it does almost nothing. The well is close enough to the stream that
pumping it takes back most of what it adds.

Run the same rule on the coastal valley, where the aquifer also meets the sea,
and the answer changes. The well now has a second source to draw on, so less of
what it pumps comes out of the stream and more of the augmented water stays
there.

## What this notebook covers

Run an operating rule on a valley with a coast, and measure how much of the
pumped water the stream keeps.

By the end of this notebook you will be able to:

- give a constant-density model a saltwater coast by raising its boundary heads
  to **equivalent freshwater heads**,
- see what that coast does to streamflow before any well is switched on,
- read streamflow from a running model and set a pumping rate from it, taking
  care to read the reach the gauge reports,
- compare an operating rule updated once per stress period with one updated every
  outer iteration, and
- measure the flow the stream keeps per unit pumped, and compare it with the
  well's stream capture fraction and the water budget.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the plotting and data libraries.
`%matplotlib inline` has to come first, or only the first figure drawn inside a
`flopy.plot.styles` context is rendered.

In [ ]:
%matplotlib inline

import pathlib as pl
import shutil

import flopy
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
from mf6_notebook_helpers import (
    coastal_ghb_data,
    find_mf6_libraries,
    length_name,
    seconds_per_day,
    to_display,
    unit,
)
from modflowapi import Callbacks, run_simulation

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the
`mf6` command-line executable. `find_mf6_libraries()` locates the library and the
`mf6` executable inside the active pixi environment; both are needed, the
executable for the ordinary baseline runs and the library for the API runs.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Build the two valleys

Both models are the **advanced** synthetic valley, which represents the valley
with streamflow routing (**SFR**) for the river, a lake (**LAK**),
unsaturated-zone flow (**UZF**), seepage drains (**DRN**) on the land surface,
and the water mover (**MVR**), which sends runoff and the water the drains
collect to the lake or the stream. SFR is what makes this notebook possible: the
rule needs a routed streamflow at a gauge to control on. The 21 stress periods
are a long steady spin-up followed by 20 annual periods, in meters and days.

The coastal valley adds a general-head boundary (**GHB**) along its southern row,
built in [mf6-coastal-ghb](mf6-coastal-ghb.ipynb), and runoff that reaches the
coast leaves the model. Seawater is denser than fresh groundwater, so a column of
it presses down harder than the same column of fresh water would. These models
carry a single constant fluid density, and the way to represent the weight of the
sea in such a model is to raise the boundary head until a column of fresh water
would press down just as hard. That raised head is the **equivalent freshwater
head**, and it grows with depth below sea level.
`coastal_ghb_data(gwf, equivalent_freshwater=True)` rebuilds the boundary with
those heads.

`build()` below loads a shipped model, sets the prediction well rate (zero for
the baselines, so the callback is the only thing that switches it on),
optionally rebuilds the coastal boundary, and writes and runs it. Both valleys
are built the same way so the only difference between them is the coast.

In [ ]:
DATA = pl.Path("../data/synthetic-valley")
MODELS = pl.Path("models")
start_date = pd.to_datetime("1962-01-01 00:00:00")
aug_start = 10  # the rule runs in the stress periods after this one


def build(source, workspace, equivalent_freshwater=False, pumping=0.0, run=True):
    """Write, and run, a synthetic-valley model with the prediction well set to
    pump `pumping` (m3/d) in the stress periods after aug_start."""
    workspace = MODELS / workspace
    if workspace.exists():
        shutil.rmtree(workspace)
    sim = flopy.mf6.MFSimulation.load(
        sim_name="sv",
        sim_ws=str(DATA / source),
        exe_name=mf6_exe,
        write_headers=False,
        verbosity_level=0,
    )
    gwf = sim.get_model("sv")

    # set the prediction well rate in every period it is listed in, so the
    # callback is the only thing that switches it on when pumping is zero
    prediction = gwf.get_package("prediction")
    rates = {}
    for kper in range(sim.tdis.nper.array):
        period = prediction.stress_period_data.get_data(kper)
        if period is None:
            continue
        period["q"] = -pumping if kper > aug_start else 0.0
        rates[kper] = period
    prediction.stress_period_data.set_data(rates)

    if equivalent_freshwater:
        spd, _ = coastal_ghb_data(gwf, equivalent_freshwater=True)
        gwf.get_package("ghb-1").stress_period_data.set_data({0: spd})

    sim.set_sim_path(str(workspace))
    sim.write_simulation(silent=True)
    if run:
        success, buff = sim.run_simulation(silent=True)
        if not success:
            raise RuntimeError("\n".join(buff[-20:]))
    return sim, workspace


coastal_sim, coastal_ws = build(
    "synthetic-valley-coastal-advanced-annual",
    "api-e-coastal-base",
    equivalent_freshwater=True,
)
inland_sim, inland_ws = build(
    "synthetic-valley-advanced-annual", "api-e-coastal-inland-base"
)

ghb = coastal_sim.get_model("sv").get_package("ghb-1").stress_period_data.get_data(0)
print(f"coastal boundary head by layer, {unit(plain=True)} above sea level")
for k in sorted({cellid[0] for cellid in ghb["cellid"]}):
    layer = ghb[[cellid[0] == k for cellid in ghb["cellid"]]]
    bhead = to_display(layer["bhead"])
    elevation = to_display(layer["elevation"])
    print(
        f"  layer {k + 1}: {bhead.min():6.3f} to {bhead.max():6.3f}"
        f"   at elevations {elevation.max():7.1f} to"
        f" {elevation.min():7.1f} {unit(plain=True)}"
    )

**What to look for.** The boundary head rises from 0.019 m (0.061 ft) in layer 1,
where the coastal cells sit 0.8 m (2.5 ft) below sea level, to between 0.94 and
1.76 m (3.08 and 5.77 ft) in layer 5, where they sit 38 to 72 m (126 to 236 ft)
down. Layer 3 has no coastal cells, because there is no clay along the coast and
its cells there are pass-through cells. Every one of those heads is above sea
level, so the sea resists outflow from the aquifer and, where the aquifer head is
low enough, drives water inland.

### Look at the coastal valley

Map the features the rule works with: the stream, the coast along the southern
row, the two production wells, and the prediction well the rule switches on. Cell
IDs are `(layer, row, column)` counting from zero, as everywhere in FloPy.

In [ ]:
gwf = coastal_sim.get_model("sv")
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
coast_rc = sorted({(cellid[1], cellid[2]) for cellid in ghb["cellid"]})

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7.5), layout="constrained")
    mv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    # heads at the end of the run; mask dry or inactive cells (flagged 1e30)
    heads = to_display(np.ma.masked_greater(gwf.output.head().get_data(), 1e29))
    cb = mv.plot_array(heads[0])
    levels = MaxNLocator(nbins=8, integer=True).tick_values(
        heads[0].min(), heads[0].max()
    )
    contours = mv.contour_array(heads[0], levels=levels, colors="black")
    ax.clabel(contours, contours.levels, fmt="%.0f", fontsize=8)
    mv.plot_bc("SFR", color="tab:cyan")
    mv.plot_bc("LAK", color="tab:blue")
    ax.plot(
        [xc[i, j] for i, j in coast_rc],
        [yc[i, j] for i, j in coast_rc],
        "s",
        color="tab:green",
        ms=6,
        label="coast",
    )
    mv.plot_bc("WELL", package=gwf.pwell, plotAll=True, color="black", kper=12)
    mv.plot_bc("WELL", package=gwf.prediction, plotAll=True, color="red", kper=12)
    ax.plot([], [], "s", color="tab:cyan", label="stream")
    ax.plot([], [], "s", color="tab:blue", label="lake")
    ax.plot([], [], "s", color="black", label="production wells")
    ax.plot([], [], "s", color="red", label="prediction well")
    fig.legend(loc="outside lower center", ncols=3, fontsize=8, frameon=False)
    # ticks at round values in the display units; the model grid is in meters
    _, xmax, _, ymax = gwf.modelgrid.extent
    for axis, vmax in ((ax.xaxis, to_display(xmax)), (ax.yaxis, to_display(ymax))):
        ticks = MaxNLocator(nbins=5, steps=[1, 2, 5, 10]).tick_values(0.0, vmax)
        ticks = ticks[ticks <= vmax]
        axis.set_ticks(ticks / to_display(1.0), labels=[f"{t:,.0f}" for t in ticks])
    ax.set_xlabel(f"x, in {length_name()}")
    ax.set_ylabel(f"y, in {length_name()}")
    fig.colorbar(cb, ax=ax, shrink=0.35, label=f"head, in {length_name()}")
    ax.set_title("Coastal synthetic valley, layer 1")

# distances from the prediction well to the nearest stream reach and the coast
_, pi, pj = gwf.prediction.stress_period_data.get_data(aug_start + 1)["cellid"][0]
stream_rc = [r["cellid"][1:] for r in gwf.sfr.packagedata.get_data()]
for label, cells in (("stream", stream_rc), ("coast", coast_rc)):
    distance = min(
        np.hypot(xc[i, j] - xc[pi, pj], yc[i, j] - yc[pi, pj]) for i, j in cells
    )
    print(
        f"prediction well to the {label}: {to_display(distance):,.0f} {unit(plain=True)}"
    )

**What to look for.** Heads in the top layer fall from the lake in the north
toward the stream and the coast in the south, with a cone around the `vc`
production well west of the stream. The prediction well (red) sits near the
southern end of the valley, 1,067 m (3,500 ft) east of the stream and 762 m
(2,500 ft) north of the coast, so both the stream and the sea are within reach of
it. That is the whole of the difference from the inland valley, where only the
stream is.

## What the coast does before any well is switched on

Read the gauge in both baselines and compare them. The gauge is the SFR
observation `riv-flow`, defined in `sv.sfr.obs` as the downstream flow of reach
17 of 18. The model's units are meters and days, so MODFLOW reports flows in
cubic meters per day (m³/d); SFR reports a flow leaving a reach as negative, so
change its sign. The rule and the tables below work in m³/d; the printed values
and the figures show stream flow per second, in m³/s, or ft³/s when
`MF6_TRAINING_UNITS` is `feet`.

The target is 0.45 m³/s (15.9 ft³/s) at the gauge, the same as in
[mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb), so the two notebooks can
be compared. The rule runs from 2016 to 2025, the stress periods after
`aug_start`.

In [ ]:
min_flow = 0.45  # target minimum streamflow at the gauge (m3/s)


def gauge_flow(sim):
    """Gauge flow per stress period in m3/d, indexed by the model's own dates."""
    observations = (
        sim.get_model("sv").sfr.output.obs().get_dataframe(start_datetime=start_date)
    )
    return -observations["RIV-FLOW"]


coastal_base = gauge_flow(coastal_sim)
inland_base = gauge_flow(inland_sim)
years = coastal_base.index  # the end of each stress period
window = slice(aug_start + 1, None)  # the periods the rule runs in
target = min_flow * seconds_per_day  # m3/d

q = unit("streamflow", plain=True)
print(f"gauge flow, {q}   start    min    mean   below target")
for label, series in (("coastal", coastal_base), ("inland", inland_base)):
    during = to_display(series.iloc[window], "streamflow")
    print(
        f"  {label:9s} {to_display(series.iloc[0], 'streamflow'):10.3f}"
        f" {during.min():6.3f} {during.mean():7.3f}"
        f"   {int((series.iloc[window] < target).sum())} of {len(during)}"
    )
difference = to_display(coastal_base - inland_base, "streamflow")
print(
    f"\n  coastal minus inland: {difference.mean():.3f} {q} on average,"
    f" {difference.min():.3f} to {difference.max():.3f}"
)

# net flow from the sea into the aquifer, from the coastal model's water budget
listing = coastal_sim.get_model("sv").output.list()
rates, _ = listing.get_dataframes(diff=True, start_datetime=None)
sea = to_display(rates["ghb"], "flow")
print(
    f"  net inflow from the sea: {sea.min():,.0f} to {sea.max():,.0f}"
    f" {unit('flow', plain=True)}"
)

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    # each value is the flow over a whole stress period, so hold it across the
    # period rather than sloping between the sampled points
    step = "steps-pre"
    ax.plot(
        years,
        to_display(coastal_base, "streamflow"),
        color="tab:green",
        lw=1.8,
        drawstyle=step,
        label="coastal valley",
    )
    ax.plot(
        years,
        to_display(inland_base, "streamflow"),
        color="tab:brown",
        lw=1.8,
        drawstyle=step,
        label="inland valley",
    )
    ax.axhline(
        to_display(target, "streamflow"), color="black", ls=":", lw=1.0, label="target"
    )
    ax.axvspan(
        years[aug_start],
        years[-1],
        color="cyan",
        alpha=0.2,
        lw=0,
        label="augmentation period",
    )
    ax.set_xlabel("year")
    ax.set_ylabel(f"gauge discharge, in {unit('streamflow')}")
    ax.set_ylim(
        0.0, 1.3 * to_display(max(coastal_base.max(), inland_base.max()), "streamflow")
    )
    ax.legend(loc="upper right", ncols=2, fontsize=8)
    ax.set_title("Baseline gauge flow, prediction well off")

**What to look for.** The coast lowers the gauge: the coastal gauge is 0.033 m³/s
(1.16 ft³/s) lower than the inland one on average over the 21 stress periods,
and lower by as much as 0.086 m³/s (3.05 ft³/s) in 2018, the wettest year. Only
in 2025, the driest year, is it higher, by 0.015 m³/s (0.53 ft³/s). The sea
takes water from the aquifer when heads are high and gives some back when they
are low: the net flow from the sea into the aquifer, printed above, runs from
-9,164 m³/d (-323,617 ft³/d), water lost to the sea, to 4,071 m³/d
(143,763 ft³/d), water gained from it.

The coast makes the driest year less dry but the others drier, and it does not
help meet the target. In the augmentation years the coastal baseline is below
0.45 m³/s in 8 of the 10 years compared to the inland valley's 7, and its lowest
year is 0.338 m³/s (11.93 ft³/s) compared to 0.323 m³/s (11.40 ft³/s).

## The operating rule

When flow at the gauge falls below `min_flow`, switch the prediction well on at a
rate proportional to the shortfall, capped at a maximum, and add that same rate
as stream inflow at the upstream end of the river. When the gauge is at or above
the target, switch the well off. The pumping rate is `GAIN` = 2.5 times the
shortfall, up to `RATE_CAP` = 20,000 m³/d (0.23 m³/s, or 8.2 ft³/s), the values
used in [mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb).

The rule reads the reach the gauge reports, `gauge_reach`, out of the SFR
downstream-flow array. The last reach, below the gauge, gains more water from
the aquifer, so a rule reading it instead would treat the target as met while
the flow at the gauge was below it.

`update_each_iteration` chooses between two ways of applying the rule:

- **False** sets the rate once per stress period, at `Callbacks.stress_period_start`,
  from the flow at the end of the previous period.
- **True** also recomputes it at the start of every outer iteration
  (`Callbacks.iteration_start`), from the flow in the current solution, so the
  rate converges along with the heads instead of lagging a period behind.

The per-iteration rate is **under-relaxed**: each outer iteration it moves only a
fraction `relax` of the way toward the rate the current flow calls for. Pumping
the well draws water from the stream, which lowers the flow the rule is reading,
so an un-relaxed rate keeps reacting to its own effect on the flow and does not
settle.

In [ ]:
gauge_reach = 16  # zero-based; the gauge is reach 17 of 18 in sv.sfr.obs

# required rate = GAIN * (shortfall below min_flow), capped at RATE_CAP
GAIN = 2.5  # dimensionless
RATE_CAP = 20000.0  # m3/d
relax = 0.5  # under-relaxation for the per-iteration update (0 < relax <= 1)

aug_rate = 0.0  # m3/d, kept across outer iterations so it can be relaxed
update_each_iteration = False


def callback_function(sim, callback_step):
    """Pump the prediction well to hold the gauge at min_flow."""
    global aug_rate
    ml = sim.sv

    def required_rate():
        """Rate (m3/d) the flow now at the gauge calls for."""
        flow = sim.mf6.get_value(sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1"))
        q = float(flow[gauge_reach]) / seconds_per_day  # m3/s
        if q < min_flow:
            return min(GAIN * (min_flow - q) * seconds_per_day, RATE_CAP)
        return 0.0

    def apply_rate(rate):
        """Pump at `rate` and put the same rate into the top of the stream."""
        ml.prediction.stress_period_data["q"] = -rate
        inflow = sim.mf6.get_value_ptr(sim.mf6.get_var_address("INFLOW", "SV", "SFR-1"))
        inflow[0] = rate

    if callback_step == Callbacks.stress_period_start and sim.kper > aug_start:
        aug_rate = required_rate()
        apply_rate(aug_rate)

    if callback_step == Callbacks.iteration_start:
        if update_each_iteration and sim.kper > aug_start:
            aug_rate += relax * (required_rate() - aug_rate)
            aug_rate = min(max(aug_rate, 0.0), RATE_CAP)
            apply_rate(aug_rate)

### Run the rule both ways

Run the coastal model twice, once with each update strategy, each in its own
workspace so the baseline output is not overwritten. `run_simulation()` drives
MODFLOW 6 through the library, calling `callback_function` at every stage.

`pumping_rate()` reads what the well actually pumped back out of the cell-by-cell
budget rather than trusting the rate the callback asked for, which checks that
the well pumped the rate the rule set.

In [ ]:
def pumping_rate(sim):
    """Prediction-well rate per stress period in m3/d, from the budget file."""
    budget = sim.get_model("sv").output.budget()
    rates = []
    for totim in budget.get_times():
        record = budget.get_data(totim=totim, paknam2="prediction")[0]
        rates.append(0.0 if len(record) == 0 else -float(record["q"][0]))
    return pd.Series(rates, index=years)


def run_rule(source, workspace, equivalent_freshwater, per_iteration):
    """Run one model under the operating rule and return (gauge flow, rate)."""
    global aug_rate, update_each_iteration
    sim, ws = build(source, workspace, equivalent_freshwater, run=False)
    aug_rate = 0.0
    update_each_iteration = per_iteration
    run_simulation(lib_name, ws, callback_function, verbose=False)
    return gauge_flow(sim), pumping_rate(sim)


coastal = {}
for label, per_iteration in (("per stress period", False), ("per iteration", True)):
    coastal[label] = run_rule(
        "synthetic-valley-coastal-advanced-annual",
        f"api-e-coastal-{per_iteration}",
        True,
        per_iteration,
    )

for label, (flow, rate) in coastal.items():
    during = to_display(flow.iloc[window], "streamflow")
    print(
        f"{label}: gauge {during.min():.3f} to {during.max():.3f} {q},"
        f" below target in {int((flow.iloc[window] < target).sum())} years;"
        f" pumping 0 to {to_display(rate.max(), 'streamflow'):.3f} {q}"
    )

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["a"], ["b"]], figsize=(9, 6), layout="constrained", sharex=True
    )
    colors = {"per stress period": "tab:orange", "per iteration": "tab:blue"}

    ax = axd["a"]
    step = "steps-pre"  # a gauge value is the flow over a whole stress period
    ax.plot(
        years,
        to_display(coastal_base, "streamflow"),
        color="tab:green",
        lw=1.8,
        drawstyle=step,
        label="baseline",
    )
    for label, (flow, _) in coastal.items():
        ax.plot(
            years,
            to_display(flow, "streamflow"),
            color=colors[label],
            lw=1.8,
            drawstyle=step,
            label=label,
        )
    ax.axhline(
        to_display(target, "streamflow"), color="black", ls=":", lw=1.0, label="target"
    )
    ax.axvspan(years[aug_start], years[-1], color="cyan", alpha=0.2, lw=0)
    ax.set_ylabel(f"gauge discharge, in {unit('streamflow')}")
    ax.set_ylim(0.0, 1.3 * to_display(coastal_base.max(), "streamflow"))
    ax.legend(loc="upper left", ncols=4, fontsize=8)

    # the rate is held constant over a stress period, so draw it as a bar in
    # the middle of that period rather than as a line between sampled points
    ax = axd["b"]
    width = 150.0  # days, the unit of a matplotlib date axis
    offsets = {"per stress period": -0.5 * width, "per iteration": 0.5 * width}
    for label, (_, rate) in coastal.items():
        ax.bar(
            mdates.date2num(years) - 182.6 + offsets[label],
            to_display(rate, "streamflow"),
            width,
            color=colors[label],
            edgecolor="black",
            lw=0.4,
            label=label,
        )
    ax.axvspan(years[aug_start], years[-1], color="cyan", alpha=0.2, lw=0, zorder=0)
    ax.set_ylabel(f"augmentation rate, in {unit('streamflow')}")
    ax.set_xlabel("year")
    ax.set_ylim(0.0, 1.2 * to_display(RATE_CAP, "streamflow"))
    for panel, ax in axd.items():
        flopy.plot.styles.heading(ax=ax, letter=panel)

**What to look for.** Panel a is the gauge and panel b the rate the rule chose.
Both strategies lift the gauge, and they fail in different ways.

Updating once per stress period (orange) sets the rate from the previous year's
flow, so the rule always responds to the previous year's flow. It pumps in 2018,
the wettest year, because 2017 was dry, lifting the gauge to 0.593 m³/s (20.94
ft³/s). It pumps nothing in 2021, a dry year, because the gauge was above the
target in 2020, and then pumps in 2022, lifting the gauge to 0.494 m³/s (17.43
ft³/s), well above the target. It pumps its most, 0.193 m³/s (6.83 ft³/s), in
2024 and nothing in 2025, the driest year, when the gauge falls to 0.332 m³/s
(11.71 ft³/s), below the 0.338 m³/s (11.93 ft³/s) with no rule at all, because
the heads near the stream are still recovering from the pumping of 2024.

Updating every outer iteration (blue) reads the flow in the solution it is part
of. Its rate never exceeds 0.117 m³/s (4.14 ft³/s), it pumps in every year the
gauge would otherwise be below the target, and its gauge flow never falls below
0.403 m³/s (14.23 ft³/s). It still leaves the gauge just below the target in 8
years, because the rate is proportional to the shortfall: the well pumps only
while the gauge is below the target, so the rule settles a little short of it. A
larger `GAIN` brings it closer. Over the augmentation period it pumps less,
16.3 million m³ (574 million ft³) compared to 18.4 million m³ (651 million ft³),
and gives a slightly lower mean gauge flow, 0.441 m³/s (15.56 ft³/s) compared to
0.444 m³/s (15.68 ft³/s), but the per-period rule pumps much of its extra water
in 2018 and 2020, years that were already above the target.

## How much of the pumped water does the stream keep?

Pumping the prediction well lowers heads near the stream, so part of the water
added at the top of the river leaks straight back into the aquifer. What the
gauge gains per unit pumped measures how effective the rule is.

Measure it as the extra water that passed the gauge over the augmentation period,
as a percentage of the water pumped, with both as volumes over the period, and
run the inland valley under the same rule to compare. The difference between the
two is the coast.

In [ ]:
inland = {}
for label, per_iteration in (("per stress period", False), ("per iteration", True)):
    inland[label] = run_rule(
        "synthetic-valley-advanced-annual",
        f"api-e-coastal-inland-{per_iteration}",
        False,
        per_iteration,
    )

# length of each stress period, days, to turn rates into volumes
perlen = np.array([p[0] for p in coastal_sim.tdis.perioddata.array])
v = unit("volume", plain=True)

rows = []
for valley, baseline, results in (
    ("coastal", coastal_base, coastal),
    ("inland", inland_base, inland),
):
    for label, (flow, rate) in results.items():
        during = flow.iloc[window]
        # volumes over the augmentation period, in millions
        lift = (
            to_display(((flow - baseline) * perlen).iloc[window].sum(), "volume")
            / 1.0e6
        )
        pumped = to_display((rate * perlen).iloc[window].sum(), "volume") / 1.0e6
        rows.append(
            {
                "valley": valley,
                "update": label,
                f"min gauge ({q})": to_display(during.min(), "streamflow"),
                f"mean gauge ({q})": to_display(during.mean(), "streamflow"),
                "years below target": int((during < target).sum()),
                f"total pumped (million {v})": pumped,
                f"total lift (million {v})": lift,
                "kept (percent of pumped)": 100.0 * lift / pumped,
            }
        )
summary = pd.DataFrame(rows).set_index(["valley", "update"])
summary.round(3)

**What to look for.** The coastal stream keeps 54.8 percent of the water pumped
under the per-period rule and 55.3 percent under the per-iteration rule. The
inland stream keeps 5.6 and 6.4 percent. The same well, the same rule, and the
same target return about nine times as much of the pumped water to the stream
once the valley has a coast.

The volumes pumped show the same thing. The inland rules pump 18.0 and 25.1
million m³ (635 and 886 million ft³) to lift the gauge by 1.01 and 1.60 million
m³ (35.8 and 56.5 million ft³); the coastal per-iteration rule pumps 16.3
million m³ (574 million ft³) to lift it by 9.0 million m³ (317 million ft³).
Inland, the rules leave the gauge below the target in 6 and 7 of the 10 years,
compared to 7 with no rule.

### Check against the stream's share of the pumped water

Water a well pumps comes from less discharge to the stream and the sea, more
inflow from them, and less water in storage. The part that comes from the stream
is the well's **stream capture fraction**. If the well takes that fraction back
out of the water the rule puts into the stream, the stream should keep about the
rest.

Measure the capture fraction directly: pump the prediction well at a constant
10,000 m³/d over the augmentation period, without putting the water into the
stream, and divide the mean fall in the gauge flow by the pumping rate.

In [ ]:
test_rate = 10000.0  # m3/d, constant over the augmentation period
capture, capture_sims = {}, {}
for valley, source, equivalent_freshwater, baseline in (
    ("coastal", "synthetic-valley-coastal-advanced-annual", True, coastal_base),
    ("inland", "synthetic-valley-advanced-annual", False, inland_base),
):
    sim, _ = build(
        source,
        f"api-e-coastal-capture-{valley}",
        equivalent_freshwater,
        pumping=test_rate,
    )
    fall = (baseline - gauge_flow(sim)).iloc[window].mean()
    capture[valley] = fall / test_rate
    capture_sims[valley] = sim

for valley, fraction in capture.items():
    kept = summary.loc[valley, "kept (percent of pumped)"]
    print(
        f"{valley}: stream capture fraction {fraction:.3f}, so the stream should"
        f" keep {100.0 * (1.0 - fraction):.1f} percent of the water pumped;"
        f" the rule kept {kept.min():.1f} to {kept.max():.1f} percent"
    )

**What to look for.** With the coast, the stream supplies 0.390 of the water the
well pumps, so the stream should keep 61.0 percent of the water the rule puts
into it; the rule kept 54.8 to 55.3 percent. Inland, the stream supplies 0.954 of
it, so the stream should keep 4.6 percent; the rule kept 5.6 to 6.4 percent. One
run at a constant pumping rate predicts the share of the pumped water the stream
keeps to within about 6 percentage points, before the rule is written.

[mf6-adj-coastal-capture](mf6-adj-coastal-capture.ipynb) gets the same stream
capture fraction for this well without any pumping run at all, from a single
backward (adjoint) solve of the coastal model.

### Where the well's water comes from

The capture fraction says how much of the well's water comes from the stream; the
water budget says where the rest comes from. Take the change in each budget term
when the well pumps at the constant rate, averaged over the augmentation period,
and divide it by the pumping rate. A positive share is water the well gets from
that source, as more inflow or less outflow, and the shares add up to 100
percent. The seepage drains are listed on their own: the water mover sends what
they collect to the lake or the stream, so less seepage also means less water
reaching the stream over land.

In [ ]:
# budget terms in the list file that make up each source
sources = {
    "sea": ("ghb",),
    "stream": ("sfr",),
    "storage": ("sto",),
    "lake": ("lak",),
    "seepage drains": ("drn",),
    "recharge reaching the water table": ("uzf-gwrch",),
}
not_sources = ["total", "in-out", "percent_discrepancy"]


def budget_rates(sim):
    """Inflow minus outflow of each budget term (m3/d), augmentation period."""
    listing = sim.get_model("sv").output.list()
    rates, _ = listing.get_dataframes(diff=True, start_datetime=None)
    return rates.iloc[window]


shares = {}
for valley, base_sim in (("coastal", coastal_sim), ("inland", inland_sim)):
    change = (budget_rates(capture_sims[valley]) - budget_rates(base_sim)).mean()
    change = change.drop(not_sources, errors="ignore")
    pumped = -change.pop("wel")
    share = {}
    for source, prefixes in sources.items():
        terms = [term for term in change.index if term.startswith(prefixes)]
        share[source] = 100.0 * change[terms].sum() / pumped
        change = change.drop(terms)
    share["other"] = 100.0 * change.sum() / pumped
    shares[valley] = share

# the list file holds single-precision numbers
sources_table = pd.DataFrame(shares).astype("float64")
sources_table.index.name = "percent of the water pumped"
sources_table.loc["total"] = sources_table.sum()
sources_table.round(1) + 0.0  # adding zero turns -0.0 into 0.0

**What to look for.** With the coast, 62.6 percent of the well's water comes from
the sea, as less fresh groundwater flowing out to it and more seawater flowing
in, and 39.0 percent comes from the stream. Storage supplies 1.9 percent, the
lake 0.5 percent, and the seepage drains none. Inland, the stream supplies 99.9
percent, storage 4.7 percent, and the lake and the seepage drains 1.4 percent
each. In both valleys less recharge reaches the water table while the well pumps,
4.1 percent of the pumping with the coast and 7.4 percent inland, and the other
sources make up that loss as well.

With the coast, the stream's share, 39.0 percent, matches the capture fraction
measured at the gauge, 0.390. Inland the two differ, 99.9 percent compared to
0.954: the budget counts the exchange with every reach, while the gauge flow
includes only the reaches above it and the water the mover delivers to them,
including what the drains collect.

## Recap

- A saltwater coast in a constant-density model is a general-head boundary at the
  **equivalent freshwater head**, which rises with depth below sea level.
- Before any well is switched on, that coast lowers the gauge in all but the
  driest year: the sea takes water from the aquifer when heads are high and
  gives some back when they are low.
- The same operating rule as
  [mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb) runs here through a
  `modflowapi` callback, reading the SFR reach the gauge reports and writing both
  the well rate and the stream inflow.
- Updating the rate every outer iteration, under-relaxed, holds the gauge steadier
  than updating once per stress period, and pumps less to do it.
- The coastal stream keeps 55 percent of the water pumped, compared with about 6
  percent inland, because the well draws most of its water from the sea rather
  than from the stream.
- The stream capture fraction of the well, 0.390, predicts 61 percent, so a
  single run at a constant rate, or an adjoint solve, can estimate the effect of
  an operating rule before the rule is written.